# MCD-rPPG Dataset - Final Corrected EDA

## All Fixes Applied

### Path Construction Fix
**Problem:** Double path prefixes like `/ecg/ecg/1020_after.json`

**Solution:** The paths in db.csv already include the subdirectory (ecg/, ppg/, etc.), so we should NOT add them again.

### Sex Column Fix  
**Problem:** Correlation fails on 'sex' column with 'F'/'M' values

**Solution:** Exclude non-numeric columns from correlation matrix

---

### Setup and Configuration

In [ ]:
# Install required packages
!pip install imageio[ffmpeg] -q
!pip install mediapipe -q
!pip install scikit-video -q

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
import json
import imageio
import skvideo.io
import cv2
from IPython.display import display, HTML
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# ============================================================================
# CORRECT CONFIGURATION
# ============================================================================
DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
DB_PATH = os.path.join(DATASET_PATH, 'db.csv')

print('=' * 80)
print('CONFIGURATION')
print('=' * 80)
print(f'DATASET_PATH: {DATASET_PATH}')
print(f'DB_PATH: {DB_PATH}')
print()
print('IMPORTANT: Paths in db.csv already include subdirectories!')
print('  db.csv["ecg"] = "ecg/1020_after.json"')
print('  Full path = DATASET_PATH + "ecg/1020_after.json"')
print('  NOT DATASET_PATH + "ecg/" + "ecg/1020_after.json"')

## 1. Load Database

In [ ]:
# Load the database
df = pd.read_csv(DB_PATH)

print('=' * 80)
print('DATABASE STRUCTURE')
print('=' * 80)
print(f'Shape: {df.shape}')
print(f'Columns: {len(df.columns)}')

print('\nColumn Names and Types:')
for col in df.columns:
    dtype = df[col].dtype
    missing = df[col].isna().sum()
    print(f'  {col:25s} : {str(dtype):10s} : {missing:4d} missing')

print('\nFirst 3 rows:')
display(df.head(3))

## 2. Prepare Metadata with CORRECT Paths

In [ ]:
# Create metadata dataframe with CORRECT full paths
print('=' * 80)
print('PREPARING METADATA WITH CORRECT PATHS')
print('=' * 80)

# Copy the original dataframe
meta_df = df.copy()

# File columns - these already include subdirectories like "ecg/", "ppg/", etc.
file_columns = ['ecg', 'ppg', 'video', 'meta', 'ppg_sync']

# Add full paths - FIXED: Just join DATASET_PATH with the path from db.csv
# Do NOT add subdirectory again!
for col in file_columns:
    if col in meta_df.columns:
        meta_df[f'{col}_full'] = meta_df[col].apply(
            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x
        )

# Add extracted metadata
meta_df['subject_id'] = meta_df['patient_id']
meta_df['condition'] = meta_df['step']
meta_df['camera_type'] = meta_df['camera']
meta_df['view_type'] = meta_df['view']

print('\nEnhanced Metadata Columns:')
enhanced_cols = [col for col in meta_df.columns if col not in df.columns or col in ['subject_id', 'condition', 'camera_type', 'view_type']]
for col in enhanced_cols:
    print(f'  {col}')

print('\nSample with full paths:')
sample_cols = ['patient_id', 'condition', 'camera_type', 'view_type', 
               'ecg_full', 'ppg_full', 'video_full', 'meta_full', 'ppg_sync_full']
display(meta_df[sample_cols].head(3))

# Verify paths are correct (no double prefixes)
print('\nVerifying paths (no double prefixes):')
for col in file_columns:
    if col in meta_df.columns:
        sample_path = meta_df[f'{col}_full'].iloc[0]
        print(f'  {col}_full: {sample_path}')
        # Check for double prefixes like "ecg/ecg/" or "video/video/"
        if 'ecg/ecg' in sample_path or 'ppg/ppg' in sample_path or 'video/video' in sample_path:
            print(f'    WARNING: Double prefix detected!')
        else:
            print(f'    OK: No double prefix')

## 3. Vital Signs Analysis

In [ ]:
# Analyze vital signs from db.csv
print('=' * 80)
print('VITAL SIGNS ANALYSIS')
print('=' * 80)

# Vital signs columns
vital_signs = ['weight', 'height', 'bmi', 'age', 
               'upper_ap', 'lower_ap', 'saturation', 'temperature', 
               'hemoglobin', 'glycated_hemoglobin', 'cholesterol', 
               'respiratory', 'rigidity', 'pulse', 'stress']

# Units for each vital sign
def get_unit(col):
    units = {
        'weight': 'kg', 'height': 'cm', 'bmi': 'kg/m2', 'age': 'years',
        'upper_ap': 'mmHg', 'lower_ap': 'mmHg', 'saturation': '%',
        'temperature': 'C', 'hemoglobin': 'g/dL', 'glycated_hemoglobin': '%',
        'cholesterol': 'mmol/L', 'respiratory': 'bpm', 'rigidity': 'm/s',
        'pulse': 'bpm', 'stress': 'score'
    }
    return units.get(col, '')

# Create statistics table
vital_stats = []
for col in vital_signs:
    if col in meta_df.columns:
        meta_df[col] = pd.to_numeric(meta_df[col], errors='coerce')
        vital_stats.append({
            'Vital Sign': col.replace('_', ' ').title(),
            'Min': f'{meta_df[col].min():.2f}',
            'Max': f'{meta_df[col].max():.2f}',
            'Mean': f'{meta_df[col].mean():.2f}',
            'Std': f'{meta_df[col].std():.2f}',
            'Unit': get_unit(col),
            'Missing': meta_df[col].isna().sum()
        })

vital_df = pd.DataFrame(vital_stats)
display(vital_df)

In [ ]:
# Plot distributions
plt.figure(figsize=(18, 12))
for i, col in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if col in meta_df.columns:
        sns.histplot(meta_df[col].dropna(), kde=True, bins=30, color='skyblue')
        plt.title(f'{col.replace("_", " ").title()}')
        plt.xlabel('')
plt.tight_layout()
plt.show()

# Plot remaining vital signs
if len(vital_signs) > 12:
    plt.figure(figsize=(12, 4))
    for i, col in enumerate(vital_signs[12:], 1):
        plt.subplot(1, len(vital_signs[12:]), i)
        if col in meta_df.columns:
            sns.histplot(meta_df[col].dropna(), kde=True, bins=30, color='salmon')
            plt.title(f'{col.replace("_", " ").title()}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix - FIXED: Exclude non-numeric columns like 'sex'
print('=' * 80)
print('CORRELATION MATRIX (Numeric columns only)')
print('=' * 80)

# Get only numeric biomarker columns
biomarker_cols = [col for col in vital_signs if col in meta_df.columns and pd.api.types.is_numeric_dtype(meta_df[col])]

print(f'Numeric biomarker columns: {biomarker_cols}')

if len(biomarker_cols) > 1:
    plt.figure(figsize=(14, 12))
    corr_matrix = meta_df[biomarker_cols].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, fmt='.2f')
    plt.title('Biomarker Correlation Matrix (Numeric Only)')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for correlation')

## 4. Video Analysis with CORRECT Paths

In [ ]:
# Analyze videos using CORRECT full paths
print('=' * 80)
print('VIDEO ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample videos with full paths
sample_videos = meta_df.dropna(subset=['video_full']).head(3)
print(f'Analyzing {len(sample_videos)} sample videos...')

for idx, row in sample_videos.iterrows():
    video_full = row['video_full']
    
    print(f'\nVideo {idx + 1}: {os.path.basename(video_full)}')
    print(f'  Full path: {video_full}')
    
    if os.path.exists(video_full):
        try:
            reader = imageio.get_reader(video_full, 'ffmpeg')
            meta_data = reader.get_meta_data()
            
            n_frames = reader.count_frames()
            fps = meta_data.get('fps', 30.0)
            duration = n_frames / fps if fps > 0 else 0
            size = meta_data.get('size', (0, 0))
            
            print(f'  Subject: {row["patient_id"]}, Condition: {row["condition"]}, Camera: {row["camera_type"]}, View: {row["view_type"]}')
            print(f'  Resolution: {size[0]}x{size[1]}')
            print(f'  FPS: {fps:.2f}')
            print(f'  Frames: {n_frames}')
            print(f'  Duration: {duration:.2f} seconds')
            
            first_frame = reader.get_next_data()
            reader.close()
            
            plt.figure(figsize=(8, 6))
            plt.imshow(first_frame)
            plt.title(f'{os.path.basename(video_full)}')
            plt.axis('off')
            plt.show()
            
        except Exception as e:
            print(f'  Error reading {video_full}: {e}')
    else:
        print(f'  NOT FOUND: {video_full}')
        print(f'  Check if path has double prefix!')

## 5. PPG Signal Analysis with CORRECT Paths

In [ ]:
# Analyze PPG signals using CORRECT full paths
print('=' * 80)
print('PPG SIGNAL ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample PPG files with full paths
sample_ppgs = meta_df.dropna(subset=['ppg_full']).head(3)
print(f'Analyzing {len(sample_ppgs)} sample PPG files...')

for idx, row in sample_ppgs.iterrows():
    ppg_full = row['ppg_full']
    
    print(f'\nPPG {idx + 1}: {os.path.basename(ppg_full)}')
    print(f'  Full path: {ppg_full}')
    
    if os.path.exists(ppg_full):
        try:
            # Handle both .npy and .PW files
            if ppg_full.endswith('.npy'):
                ppg_signal = np.load(ppg_full)
            elif ppg_full.endswith('.PW'):
                ppg_signal = np.loadtxt(ppg_full)
            else:
                ppg_signal = np.load(ppg_full)
            
            print(f'  Subject: {row["patient_id"]}, Condition: {row["condition"]}')
            print(f'  Shape: {ppg_signal.shape}')
            print(f'  Dtype: {ppg_signal.dtype}')
            print(f'  Duration: {len(ppg_signal) / 100:.2f} seconds (at 100Hz)')
            print(f'  Min: {ppg_signal.min():.2f}, Max: {ppg_signal.max():.2f}, Mean: {ppg_signal.mean():.2f}')
            
            plt.figure(figsize=(15, 4))
            plt.plot(ppg_signal[:1000], color='red', linewidth=1)
            plt.title(f'PPG Signal - {os.path.basename(ppg_full)}')
            plt.xlabel('Sample Index')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
            
        except Exception as e:
            print(f'  Error loading {ppg_full}: {e}')
    else:
        print(f'  NOT FOUND: {ppg_full}')
        print(f'  Check if path has double prefix!')

## 6. PPG_Sync Analysis with CORRECT Paths

In [ ]:
# Analyze PPG sync files using CORRECT full paths
print('=' * 80)
print('PPG_SYNC ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample PPG sync files with full paths
sample_ppg_syncs = meta_df.dropna(subset=['ppg_sync_full']).head(3)
print(f'Analyzing {len(sample_ppg_syncs)} sample PPG sync files...')

for idx, row in sample_ppg_syncs.iterrows():
    ppg_sync_full = row['ppg_sync_full']
    
    print(f'\nPPG Sync {idx + 1}: {os.path.basename(ppg_sync_full)}')
    print(f'  Full path: {ppg_sync_full}')
    
    if os.path.exists(ppg_sync_full):
        try:
            if ppg_sync_full.endswith('.txt'):
                df_sync = pd.read_csv(ppg_sync_full, sep='\s+', header=None)
            elif ppg_sync_full.endswith('.npy'):
                df_sync = np.load(ppg_sync_full)
                df_sync = pd.DataFrame(df_sync)
            else:
                df_sync = pd.read_csv(ppg_sync_full)
            
            sync_rows = len(df_sync)
            print(f'  Rows: {sync_rows}')
            print(f'  Shape: {df_sync.shape}')
            print(f'  First few values:')
            display(df_sync.head())
            
        except Exception as e:
            print(f'  Error reading {ppg_sync_full}: {e}')
    else:
        print(f'  NOT FOUND: {ppg_sync_full}')
        print(f'  Check if path has double prefix!')

## 7. ECG Signal Analysis (500Hz) with CORRECT Paths

In [ ]:
# Analyze ECG signals using CORRECT full paths
print('=' * 80)
print('ECG SIGNAL ANALYSIS (500Hz) WITH CORRECT PATHS')
print('=' * 80)

# Get sample ECG files with full paths
sample_ecgs = meta_df.dropna(subset=['ecg_full']).head(3)
print(f'Analyzing {len(sample_ecgs)} sample ECG files...')

for idx, row in sample_ecgs.iterrows():
    ecg_full = row['ecg_full']
    
    print(f'\nECG {idx + 1}: {os.path.basename(ecg_full)}')
    print(f'  Full path: {ecg_full}')
    
    if os.path.exists(ecg_full):
        try:
            # Check file extension
            if ecg_full.endswith('.json'):
                with open(ecg_full, 'r') as f:
                    ecg_data = json.load(f)
                
                print(f'  Type: JSON')
                if 'signal' in ecg_data:
                    ecg_signal = np.array(ecg_data['signal'])
                    sampling_rate = ecg_data.get('sampling_rate', 500)
                    print(f'  Signal length: {len(ecg_signal)}')
                    print(f'  Sampling rate: {sampling_rate} Hz')
                    print(f'  Duration: {len(ecg_signal) / sampling_rate:.2f} seconds')
                    
                    plt.figure(figsize=(15, 4))
                    plt.plot(ecg_signal[:1000], color='green', linewidth=1)
                    plt.title(f'ECG Signal - {os.path.basename(ecg_full)} (500Hz)')
                    plt.xlabel('Sample Index')
                    plt.ylabel('ECG Value')
                    plt.grid(True, alpha=0.3)
                    plt.show()
            else:
                ecg_signal = np.load(ecg_full)
                print(f'  Type: NPY')
                print(f'  Shape: {ecg_signal.shape}')
                print(f'  Duration: {len(ecg_signal) / 500:.2f} seconds (at 500Hz)')
                
                plt.figure(figsize=(15, 4))
                plt.plot(ecg_signal[:1000], color='green', linewidth=1)
                plt.title(f'ECG Signal - {os.path.basename(ecg_full)} (500Hz)')
                plt.xlabel('Sample Index')
                plt.ylabel('ECG Value')
                plt.grid(True, alpha=0.3)
                plt.show()
            
        except Exception as e:
            print(f'  Error loading {ecg_full}: {e}')
    else:
        print(f'  NOT FOUND: {ecg_full}')
        print(f'  Check if path has double prefix!')

## 8. Video vs PPG_Sync Comparison

In [ ]:
# Compare video frames with PPG sync data
print('=' * 80)
print('VIDEO VS PPG_SYNC COMPARISON')
print('=' * 80)

# Get sample pairs
sample_df = meta_df.dropna(subset=['video_full', 'ppg_sync_full']).head(3)

for idx, row in sample_df.iterrows():
    video_full = row['video_full']
    ppg_sync_full = row['ppg_sync_full']
    
    print(f'\nSample {idx + 1}: Subject={row["patient_id"]}, Condition={row["condition"]}, Camera={row["camera_type"]}')
    print(f'  Video path: {video_full}')
    print(f'  PPG sync path: {ppg_sync_full}')
    
    # Analyze video
    if os.path.exists(video_full):
        try:
            reader = imageio.get_reader(video_full, 'ffmpeg')
            n_frames = reader.count_frames()
            fps = reader.get_meta_data().get('fps', 30.0)
            video_duration = n_frames / fps if fps > 0 else 0
            reader.close()
            print(f'  Video: {n_frames} frames, {fps:.2f} FPS, {video_duration:.2f}s')
        except Exception as e:
            print(f'  Error reading video: {e}')
    else:
        print(f'  Video NOT FOUND: {video_full}')
    
    # Analyze PPG sync
    if os.path.exists(ppg_sync_full):
        try:
            if ppg_sync_full.endswith('.txt'):
                df_sync = pd.read_csv(ppg_sync_full, sep='\s+', header=None)
            elif ppg_sync_full.endswith('.npy'):
                df_sync = np.load(ppg_sync_full)
                df_sync = pd.DataFrame(df_sync) if df_sync.ndim > 1 else pd.DataFrame(df_sync.reshape(-1, 1))
            else:
                df_sync = pd.read_csv(ppg_sync_full)
            
            sync_rows = len(df_sync)
            print(f'  PPG Sync: {sync_rows} rows')
            
            # Compare
            if n_frames > 0 and sync_rows > 0:
                diff = n_frames - sync_rows
                if diff == 0:
                    print('  OK PERFECT MATCH: Video frames == PPG sync rows')
                else:
                    print(f'  WARNING MISMATCH: Difference of {abs(diff)} rows ({diff:+d})')
        except Exception as e:
            print(f'  Error reading PPG sync: {e}')
    else:
        print(f'  PPG sync NOT FOUND: {ppg_sync_full}')
    
    print()

## 9. Landmarks Analysis with MediaPipe - 3 View Cases

In [ ]:
# Initialize MediaPipe and analyze 3 different view cases
print('=' * 80)
print('LANDMARKS ANALYSIS WITH MEDIAPIPE - 3 VIEW CASES')
print('=' * 80)

try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    print('OK MediaPipe available!')
    
    # Define ROI landmarks
    rois = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 53)),
        'right_eye': list(range(252, 283)),
        'nose': list(range(1, 21)) + list(range(195, 221)),
        'mouth': list(range(60, 81)) + list(range(290, 321)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }
    
    # Get 3 different view cases
    view_cases = meta_df['view_type'].unique()[:3]
    print(f'Analyzing {len(view_cases)} different view cases: {view_cases}')
    
    # Initialize MediaPipe
    base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
        num_faces=1)
    detector = vision.FaceLandmarker.create_from_options(options)
    
    # Process each view case
    for view_case in view_cases:
        print(f'\nVIEW CASE: {view_case}')
        
        sample = meta_df[meta_df['view_type'] == view_case].dropna(subset=['video_full']).head(1)
        
        if len(sample) > 0:
            row = sample.iloc[0]
            video_full = row['video_full']
            
            print(f'  Video path: {video_full}')
            
            if os.path.exists(video_full):
                try:
                    reader = imageio.get_reader(video_full, 'ffmpeg')
                    first_frame = reader.get_next_data()
                    reader.close()
                    
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=first_frame)
                    detection_result = detector.detect(mp_image)
                    
                    if detection_result.face_landmarks:
                        landmarks = detection_result.face_landmarks[0]
                        print(f'  Detected {len(landmarks)} landmarks')
                        
                        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
                        
                        # Full landmarks
                        ax1.imshow(first_frame)
                        ax1.scatter([p.x * first_frame.shape[1] for p in landmarks],
                                  [p.y * first_frame.shape[0] for p in landmarks],
                                  s=1, c='red', alpha=0.5)
                        ax1.set_title(f'{view_case} - All Landmarks')
                        ax1.axis('off')
                        
                        # ROI boxes (24x24)
                        ax2.imshow(first_frame)
                        for roi_name, roi_landmarks in rois.items():
                            if roi_landmarks and roi_name != 'full_face':
                                valid_indices = [i for i in roi_landmarks if i < len(landmarks)]
                                if valid_indices:
                                    roi_points = [landmarks[i] for i in valid_indices]
                                    x_coords = [p.x * first_frame.shape[1] for p in roi_points]
                                    y_coords = [p.y * first_frame.shape[0] for p in roi_points]
                                    center_x = int(np.mean(x_coords))
                                    center_y = int(np.mean(y_coords))
                                    
                                    rect = patches.Rectangle(
                                        (center_x - 12, center_y - 12), 24, 24,
                                        linewidth=2, edgecolor='cyan', facecolor='none')
                                    ax2.add_patch(rect)
                                    ax2.text(center_x - 12, center_y - 15, roi_name,
                                           color='cyan', fontsize=8,
                                           bbox=dict(facecolor='cyan', alpha=0.5))
                        
                        ax2.set_title(f'{view_case} - ROI Boxes (24x24)')
                        ax2.axis('off')
                        plt.suptitle(f'View: {view_case}')
                        plt.tight_layout()
                        plt.show()
                    else:
                        print('  No face detected')
                except Exception as e:
                    print(f'  Error: {e}')
            else:
                print(f'  Video NOT FOUND: {video_full}')
        else:
            print(f'  No sample for view: {view_case}')
    
    print('\nLandmark analysis complete!')
    
except ImportError as e:
    print(f'MediaPipe not available: {e}')
    print('Install with: pip install mediapipe')

## 10. Summary and Key Fixes

In [ ]:
print('=' * 80)
print('SUMMARY AND KEY FIXES')
print('=' * 80)

print('DATABASE STRUCTURE:')
print(f'  Total entries: {len(meta_df)}')
print(f'  Unique subjects: {meta_df["patient_id"].nunique()}')
print(f'  Conditions: {meta_df["condition"].unique()}')
print(f'  Camera types: {meta_df["camera_type"].unique()}')
print(f'  View types: {meta_df["view_type"].unique()}')

print()
print('KEY FIXES APPLIED:')
print('  FIXED: Path construction - NO double prefixes')
print('         db.csv paths already include subdirectories (ecg/, ppg/, etc.)')
print('         Full path = DATASET_PATH + path_from_db.csv')
print()
print('  FIXED: Correlation matrix - Excluded non-numeric columns (sex)')
print('         Only numeric biomarker columns used for correlation')
print()
print('  FIXED: PPG loading - Handles both .npy and .PW files')
print('  FIXED: Video frame count - Uses reader.count_frames()')
print('  FIXED: ROI boxes - 24x24 for each view case')

print()
print('PATH EXAMPLES:')
print('  db.csv["ecg"]:     "ecg/1020_after.json"')
print('  Full path:         DATASET_PATH + "ecg/1020_after.json"')
print(f'  = {DATASET_PATH}ecg/1020_after.json')
print()
print('  WRONG (double):    DATASET_PATH + "ecg/" + "ecg/1020_after.json"')
print(f'  = {DATASET_PATH}ecg/ecg/1020_after.json')
print()
print('NEXT STEPS:')
print('  1. Verify all paths are correct (no double prefixes)')
print('  2. Process videos with imageio[ffmpeg]')
print('  3. Load PPG/ECG signals with correct methods')
print('  4. Extract landmarks with MediaPipe')
print('  5. Align signals using timestamps')